# Figure 3C: CVG to Uns.

Set the paths marked below before running this notebook.

The input is `tree_results-1.pkl`, the accepted result package prepared by
`04_tree-1.ipynb`. It contains the tree display and the heatmap data originally
stored in `fig3CG_results.h5` by the `04_tree.ipynb` analysis.

This panel uses 697 gene rows and 2,332 cells. It reuses the saved fitted
expression, gene and cell order, per-row scaling, color strips and gene labels.
The `render_composites` function is copied from `04_tree-1.ipynb`, and its panel loop
is restricted to C. The two output paths produce this panel's PDF and PNG.


In [ ]:
#1. =========== Paths and panel selection
from pathlib import Path
import hashlib
import pickle
import warnings
import numpy as np
import pandas as pd

ROOT = Path(".")  # Set the project root before using these path placeholders.
# Set the data file path. Uses tree_results-1.pkl prepared by 04_tree-1.ipynb.
# Its heatmaps preserve the fits from fig3CG_results.h5 and the 04_tree.ipynb analysis.
RESULT = ROOT
PDF_FILE = ROOT  # Set the output PDF file path for Fig3C.
PNG_FILE = ROOT  # Set the output PNG file path for Fig3C.
OUTPUT_DIR = ROOT  # Set the output directory path.
WORK = ROOT  # Set the temporary working directory path.
PDFTOPPM_FILE = ROOT  # Set the pdftoppm executable file path.
FONT_FILES = {
    "Arial": ROOT,  # Set the Arial regular TTF file path.
    "Arial-Bold": ROOT,  # Set the Arial bold TTF file path.
    "Arial-Italic": ROOT,  # Set the Arial italic TTF file path.
}

PANEL = "C"
EXPECTED_GENES = 697
EXPECTED_CELLS = 2332


In [ ]:
#2. =========== Read the saved tree and heatmap
with warnings.catch_warnings(), RESULT.open("rb") as handle:
    warnings.simplefilter("ignore", DeprecationWarning)
    bundle = pickle.load(handle)
assert "publication_display" in bundle and "heatmaps" in bundle
publication_display = bundle["publication_display"]
heatmap = bundle["heatmaps"][PANEL]

def array_sha(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()

snapshot = bundle["accepted_display_snapshot"]
assert array_sha(publication_display["cell_xy"]) == snapshot["cell_xy_sha256"]
assert array_sha(publication_display["node_y"]) == snapshot["node_y_sha256"]
assert array_sha(bundle["model"]["F"]) == snapshot["source_F_sha256"]
assert array_sha(bundle["model"]["B"]) == snapshot["source_B_sha256"]
assert array_sha(bundle["projection"]["t"]) == snapshot["original_t_sha256"]
assert heatmap["rgb"].shape == (EXPECTED_GENES, EXPECTED_CELLS, 3)
assert len(heatmap["genes"]) == EXPECTED_GENES
assert len(heatmap["obs"]) == EXPECTED_CELLS
assert array_sha(heatmap["rgb"]) == heatmap["rgb_sha256"]
assert array_sha(heatmap["normalized"]) == heatmap["normalized_sha256"]


In [ ]:
#3. =========== Tree and heatmap renderer
def render_composites(bundle, display, heatmaps, outdir, work):
    """Draw the selected heatmap with its tree, color strips and gene labels."""
    from pathlib import Path
    import hashlib
    import io
    import math
    import re
    import subprocess
    import sys
    import numpy as np
    from PIL import Image

    # Use the same physical geometry as the original five-panel renderer.
    DPI = 600
    UNDERPAINT_PT = .2  # Preserve the accepted PDF seam treatment.
    SIZE = (5.45, 4.6)
    HEAT_SHIFT = 1.65
    MATRIX = (HEAT_SHIFT+.20*3.8, .24*4.6, .50*3.8, .58*4.6)
    TREE_LEFT = .20
    TREE_HEIGHT = MATRIX[3]
    TREE_WIDTH = TREE_HEIGHT*2.8/4.845
    TREE_SCALE = TREE_HEIGHT/4.845
    STRIP_HEIGHT, STRIP_GAP = .014*4.6, .004*4.6
    XMIN, XMAX = -.5, 3.5
    TREE_POINT_DIAMETER = math.sqrt(1.4)*TREE_SCALE
    TREE_ALPHA = .88
    STAGES = ['E9.5', 'E10.5', 'E11.5', 'E13.5', 'E14.5', 'E15.5', 'E16.5', 'E17.5', 'E18.5']
    STAGE_COLORS = ['#f5b888', '#f2a985', '#ee9982', '#e58780', '#cd717e', '#b55c7d', '#914679', '#653175', '#3a1c71']
    FATE_BY_PANEL = {'D': 'Ia', 'E': 'Ib', 'F': 'Ic', 'G': 'II'}
    from reportlab.pdfgen import canvas
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.lib.colors import Color, HexColor
    for name, filename in FONT_FILES.items():
        if name not in pdfmetrics.getRegisteredFontNames():
            pdfmetrics.registerFont(TTFont(name, str(filename)))
    outdir, work = Path(outdir), Path(work)
    outdir.mkdir(parents=True, exist_ok=True); work.mkdir(parents=True, exist_ok=True)
    pdftoppm = PDFTOPPM_FILE
    assert pdftoppm.is_file()
    obs = bundle['obs']
    xy = np.asarray(display['cell_xy'], float)
    node_y = np.asarray(display['node_y'], float)
    node_x = display['node_x']
    segment_x = np.asarray(display['segment_x'], float)
    assert xy.shape == (len(obs), 2) and np.isfinite(xy).all()
    assert len(segment_x) == len(bundle['segments'])
    maximum = float(node_y.max())
    ymin, ymax = -.045*maximum, 1.32*maximum
    graph_x = lambda value: 1.5+.5*(np.asarray(value)-1.5)
    original_xy_sha = hashlib.sha256(xy.tobytes()).hexdigest()
    width, height = np.array(SIZE)*72
    artifacts, scenes, image_boxes, mappings, stripped_symbols = {}, {}, {}, {}, {}

    def as_rgb8(value):
        a = np.asarray(value)
        assert a.ndim >= 2 and a.shape[-1] == 3 and np.isfinite(a).all()
        if a.dtype == np.uint8:
            return np.ascontiguousarray(a)
        assert a.min() >= 0
        if a.max() <= 1:
            return np.ascontiguousarray(np.rint(a*255).astype(np.uint8))
        assert a.max() <= 255
        return np.ascontiguousarray(np.rint(a).astype(np.uint8))

    def vector_matrix(c, matrix, box):
        """Restore editable same-color runs without resampling rows or columns."""
        rows, columns = matrix.shape[:2]
        keys = ((matrix[..., 0].astype(np.uint32) << 16) |
                (matrix[..., 1].astype(np.uint32) << 8) | matrix[..., 2].astype(np.uint32))
        start = np.ones((rows, columns), dtype=bool)
        start[:, 1:] = keys[:, 1:] != keys[:, :-1]
        rr, cc = np.nonzero(start)
        next_c, next_r = np.r_[cc[1:], columns], np.r_[rr[1:], rows]
        end = np.where(next_r == rr, next_c, columns)
        run_keys = keys[rr, cc]
        order = np.argsort(run_keys, kind='stable')
        sorted_keys = run_keys[order]
        splits = np.r_[0, np.flatnonzero(sorted_keys[1:] != sorted_keys[:-1])+1, len(order)]
        mx, my, mw, mh = map(float, box)
        dx, dy = mw/columns, mh/rows
        groups = []
        for a, b in zip(splits[:-1], splits[1:]):
            chosen = order[a:b]
            key = int(sorted_keys[a])
            color = ((key >> 16)/255., ((key >> 8) & 255)/255., (key & 255)/255.)
            rectangles = '\n'.join(
                f'{mx+cc[i]*dx:.8f} {my+(rows-rr[i]-1)*dy:.8f} '
                f'{(end[i]-cc[i])*dx:.8f} {dy:.8f} re' for i in chosen)
            groups.append((color, rectangles))
        c._code.append('%%HEATMAP_VECTOR_BEGIN')
        c.saveState()
        c.setLineJoin(0)
        c.setLineCap(0)
        c.setLineWidth(UNDERPAINT_PT)
        c._code.append('%%HEATMAP_UNDERPAINT_BEGIN')
        for color, rectangles in groups:
            channels = ' '.join(f'{v:.10f}' for v in color)
            c._code.append(f'{channels} rg\n{channels} RG\n{rectangles}\nB')
        c._code.append('%%HEATMAP_MAIN_BEGIN')
        for color, rectangles in groups:
            channels = ' '.join(f'{v:.10f}' for v in color)
            c._code.append(f'{channels} rg\n{rectangles}\nf')
        c._code.append('%%HEATMAP_MAIN_END')
        c.restoreState()
        c._code.append('%%HEATMAP_VECTOR_END')
        return dict(runs=int(len(rr)), colors=len(groups), underpaint_pt=UNDERPAINT_PT,
                    rgb_sha256=hashlib.sha256(matrix.tobytes()).hexdigest())

    def rgb(value):
        if isinstance(value, str):
            return HexColor(value)
        a = np.asarray(value, float)
        if a.max() > 1:
            a = a/255
        return Color(*a)

    def text(c, value, x, y, size=7., font='Arial', fill='#252525', align='left', role='label'):
        lines = str(value).split('\n')
        widths = [pdfmetrics.stringWidth(line, font, size) for line in lines]
        asc, desc = pdfmetrics.getAscentDescent(font, size)
        leading = size*1.18
        obj = c.beginText(); obj.setFont(font, size); obj.setFillColor(rgb(fill))
        for i, (line, line_width) in enumerate(zip(lines, widths)):
            dx = 0 if align == 'left' else (-line_width/2 if align == 'center' else -line_width)
            obj.setTextOrigin(x+dx, y-i*leading); obj.textOut(line)
        c.drawText(obj)
        left = x if align == 'left' else x-max(widths)/(2 if align == 'center' else 1)
        c._scene.append(dict(text=str(value), role=role, font=font,
                            bbox=[left, y+desc-(len(lines)-1)*leading, left+max(widths), y+asc]))

    def line(c, x0, y0, x1, y1, fill='#777777', weight=.45):
        c.setStrokeColor(rgb(fill)); c.setLineWidth(weight); c.line(x0, y0, x1, y1)

    def circle(c, x, y, radius, fill, stroke=None, weight=.35):
        k = radius*4*(math.sqrt(2)-1)/3
        p = c.beginPath(); p.moveTo(x+radius, y)
        p.curveTo(x+radius, y+k, x+k, y+radius, x, y+radius)
        p.curveTo(x-k, y+radius, x-radius, y+k, x-radius, y)
        p.curveTo(x-radius, y-k, x-k, y-radius, x, y-radius)
        p.curveTo(x+k, y-radius, x+radius, y-k, x+radius, y); p.close()
        c.setFillColor(rgb(fill))
        if stroke is not None:
            c.setStrokeColor(rgb(stroke)); c.setLineWidth(weight)
        c.drawPath(p, fill=1, stroke=int(stroke is not None))

    def tree_map(x, y):
        return ((TREE_LEFT+TREE_WIDTH*(np.asarray(x)-XMIN)/(XMAX-XMIN))*72,
                (MATRIX[1]+TREE_HEIGHT*(ymax-np.asarray(y))/(ymax-ymin))*72)

    def graph_map(x, y):
        return tree_map(graph_x(x), y)

    def vector_strip(c, values, box, frame=True):
        values = as_rgb8(values)
        assert values.ndim == 2
        x, y, w, h = np.array(box)*72
        changes = np.r_[0, np.flatnonzero(np.any(values[1:] != values[:-1], axis=1))+1, len(values)]
        for start, end in zip(changes[:-1], changes[1:]):
            x0, x1 = x+w*start/len(values), x+w*end/len(values)
            c.setFillColor(rgb(values[start]))
            overlap = min(.005, x+w-x1)
            c.rect(x0, y, x1-x0+overlap, h, fill=1, stroke=0)
        if frame:
            c.setStrokeColor(rgb('#333333')); c.setLineWidth(.45); c.rect(x, y, w, h, fill=0, stroke=1)

    def clean_symbol(value):
        return re.sub(r'[\s*\u2020\u2021\u00b0]+$', '', str(value).strip())

    for panel in (PANEL,):
        item = heatmaps[panel]
        matrix = as_rgb8(item['rgb'])
        assert matrix.ndim == 3
        n_genes, n_cells = matrix.shape[:2]
        cells = item['obs']
        labels = item['labels']
        assert len(cells) == n_cells
        cell_ids = cells['cell_id'].astype(str).to_numpy()
        assert len(np.unique(cell_ids)) == n_cells
        positions = obs.index.get_indexer(cell_ids)
        assert np.all(positions >= 0)
        age_rgb = as_rgb8(item['stage_strip_rgb'])
        time_rgb = as_rgb8(item['pseudotime_strip_rgb'])
        assert age_rgb.shape == time_rgb.shape == (n_cells, 3)
        selected_mask = np.zeros(len(obs), bool)
        selected_mask[positions] = True
        if 'tree_mask' in item:
            np.testing.assert_array_equal(selected_mask, np.asarray(item['tree_mask'], bool))
        tree_rgb = np.full((len(obs), 3), 255, np.uint8)
        tree_rgb[positions] = time_rgb
        if 'tree_rgb' in item:
            np.testing.assert_array_equal(tree_rgb, np.asarray(item['tree_rgb'], np.uint8))
        if 'tree_cell_ids' in item:
            assert np.array_equal(cell_ids, np.asarray(item['tree_cell_ids']).astype(str))
        if panel != 'C':
            expected = np.asarray(bundle['figures'][FATE_BY_PANEL[panel]]['mask'], bool)
            assert set(obs.index[expected]) == set(cell_ids)
        else:
            assert n_cells == 2332, 'Panel C must retain the original 2,332-cell set'
        pdf, png = PDF_FILE, PNG_FILE
        c = canvas.Canvas(str(pdf), pagesize=(width, height), invariant=1, pageCompression=1,
                          initialFontName='Arial', initialFontSize=7)
        c.setTitle(f'Fig3{panel}: tree and gene dynamics')
        c.setAuthor('Developmental map of auditory neurons')
        c.setSubject('Vector RGB heatmap rectangles; unchanged tree, strips, leaders and live Arial gene labels')
        c._scene = []

        # The full accepted display is reused for all five trees at one uniform scale.
        for k, seg in enumerate(bundle['segments']):
            start, end = int(seg['start']), int(seg['end'])
            x0, y0 = graph_map(segment_x[k], node_y[start])
            x1, y1 = graph_map(segment_x[k], node_y[end])
            line(c, x0, y0, x1, y1, '#A7AAAC', .65*TREE_SCALE)
            if node_x[start] != segment_x[k]:
                xp, yp = graph_map(node_x[start], node_y[start])
                line(c, xp, yp, x0, y0, '#A7AAAC', .65*TREE_SCALE)
        c.setFillAlpha(TREE_ALPHA)
        # Heatmap pseudotime colors are aligned to the same cell IDs, never to row rank alone.
        source_order = np.asarray(bundle.get('plot_order', np.arange(len(obs))), int)
        for cell in source_order[selected_mask[source_order]]:
            xp, yp = tree_map(*xy[cell])
            circle(c, float(xp), float(yp), TREE_POINT_DIAMETER/2, tree_rgb[cell])
        c.setFillAlpha(1); c.setStrokeAlpha(1)
        root = int(bundle['projection']['root'])
        xp, yp = graph_map(node_x[root], node_y[root])
        circle(c, xp, yp, 1.0, '#FFFFFF', '#555555', .35)
        text(c, 'Root', xp, yp+4, 5.8, align='center', role='tree_label')
        if 'vgn_tip' in bundle:
            vg = int(bundle['vgn_tip'])
            xp, yp = graph_map(node_x[vg], node_y[vg])
            text(c, 'VGN candidate-\nenriched', xp, yp-6, 5.1, align='center', role='tree_label')
        for fate in ('Ib', 'Ic', 'Ia', 'II'):
            tip = int(bundle['figures'][fate]['tip'])
            xp, yp = graph_map(node_x[tip], node_y[tip])
            fill = '#333333' if FATE_BY_PANEL.get(panel) == fate else '#999999'
            circle(c, xp, yp, .9, '#FFFFFF', fill, .3)
            text(c, fate, xp, yp-7, 5.9, font='Arial-Bold' if FATE_BY_PANEL.get(panel) == fate else 'Arial',
                 fill=fill, align='center', role='tree_label')
        if panel in FATE_BY_PANEL:
            start = int(bundle['figures'][FATE_BY_PANEL[panel]]['start'])
            xp, yp = graph_map(node_x[start], node_y[start])
            circle(c, xp, yp, 1.25, '#FFFFFF', '#555555', .4)
        tree_heading = 'Original common cell set' if panel == 'C' else FATE_BY_PANEL[panel]+' path'
        text(c, tree_heading, TREE_LEFT*72, (MATRIX[1]+MATRIX[3]+.20)*72, 7.5,
             font='Arial-Bold', role='tree_title')
        circle(c, TREE_LEFT*72+2, 62, 1.8, time_rgb[len(time_rgb)//2])
        text(c, f'Selected cells ({n_cells:,})', TREE_LEFT*72+9, 60, 5.8, role='legend')
        line(c, TREE_LEFT*72, 49, TREE_LEFT*72+5, 49, '#A7AAAC', .6)
        text(c, 'Other tree segments', TREE_LEFT*72+9, 47, 5.8, role='legend')

        # Preserve the frozen RGB matrix as editable rectangle subpaths.
        mx, my, mw, mh = np.array(MATRIX)*72
        c.setFillAlpha(1); c.setStrokeAlpha(1)
        vector_report = vector_matrix(c, matrix, (mx, my, mw, mh))
        c.setStrokeColor(rgb('#333333')); c.setLineWidth(.5); c.rect(mx, my, mw, mh, fill=0, stroke=1)
        image_boxes[panel] = dict(x=mx, y=my, width=mw, height=mh, pixels=[n_cells, n_genes],
                                  color_space='DeviceRGB', alpha=False, image_count=0, vector_report=vector_report,
                                  rgb_sha256=hashlib.sha256(matrix.tobytes()).hexdigest())
        time_box = (MATRIX[0], MATRIX[1]+MATRIX[3]+STRIP_GAP, MATRIX[2], STRIP_HEIGHT)
        age_box = (MATRIX[0], MATRIX[1]+MATRIX[3]+2*STRIP_GAP+STRIP_HEIGHT, MATRIX[2], STRIP_HEIGHT)
        vector_strip(c, time_rgb, time_box); vector_strip(c, age_rgb, age_box)
        text(c, 'Pseudotime', mx-3, (time_box[1]+STRIP_HEIGHT/2)*72-1.7, 5.6, align='right', role='strip_label')
        text(c, 'Age', mx-3, (age_box[1]+STRIP_HEIGHT/2)*72-1.7, 5.6, align='right', role='strip_label')

        # One complete live italic text object per gene, with vector leader lines.
        label_rows = []
        if hasattr(labels, 'iterrows'):
            iterator = [row for _, row in labels.iterrows()]
        else:
            iterator = labels
        for row in iterator:
            symbol = clean_symbol(row['symbol'])
            index = int(row['display_row'])
            assert 0 <= index < n_genes and symbol
            label_rows.append((index, symbol, str(row['symbol'])))
        label_rows.sort(key=lambda x: x[0])
        assert len({x[1] for x in label_rows}) == len(label_rows)
        anchors = np.array([(row+.5)*mh/n_genes for row, _, _ in label_rows])
        locations = anchors.copy()
        min_gap, padding = 9., 4.
        if len(locations):
            for i in range(1, len(locations)):
                locations[i] = max(locations[i], locations[i-1]+min_gap)
            locations[-1] = min(locations[-1], mh-padding)
            for i in range(len(locations)-2, -1, -1):
                locations[i] = min(locations[i], locations[i+1]-min_gap)
            if locations[0] < padding:
                locations += padding-locations[0]
            assert locations[-1] <= mh-padding+1e-9
        for (row, symbol, original), anchor, target in zip(label_rows, anchors, locations):
            ya, yt = my+mh-anchor, my+mh-target
            line(c, mx+mw+.8, ya, mx+mw+4.5, ya, '#555555', .4)
            line(c, mx+mw+4.5, ya, mx+mw+10.5, yt, '#555555', .4)
            text(c, symbol, mx+mw+12.5, yt-2.3, 7., font='Arial-Italic', role='gene')
            if symbol != original:
                stripped_symbols.setdefault(panel, []).append(dict(original=original, displayed=symbol))
        text(c, panel.lower(), 9, height-15, 14, font='Arial-Bold', role='panel_letter')
        text(c, str(item.get('title', FATE_BY_PANEL.get(panel, 'Common early path'))), mx, height-14,
             10, font='Arial-Bold', role='title')
        stage_values = cells['stage'].astype(str) if 'stage' in cells else cells['stage_standardized'].astype(str)
        present = [s for s in STAGES if s in set(stage_values)]
        stage_text = (present[0]+' - '+present[-1]) if present else 'Observed stages'
        text(c, f'{stage_text} | {n_cells:,} cells | {n_genes:,} genes', mx, height-29, 6.4, fill='#555555', role='subtitle')
        for fraction in (0., .5, 1.):
            x = mx+mw*fraction
            line(c, x, my, x, my-2.2, '#333333', .45)
            text(c, str(int(round(n_cells*fraction))), x, my-9, 5.7, align='center', role='tick')
        text(c, 'Cells ordered by original pseudotime', mx+mw/2, my-20, 6.2, align='center', role='axis')

        # The two continuous colorbars are also made from independent vector strips.
        expr_cmap = item.get('expression_cmap_rgb')
        if expr_cmap is None:
            from matplotlib import colormaps
            expr_cmap = colormaps['RdBu_r'](np.linspace(0, 1, 256))[:, :3]
        expr_box = (MATRIX[0], .565, 1.12, .05)
        vector_strip(c, expr_cmap, expr_box, frame=False)
        low, high = map(float, item['color_limits'])
        text(c, 'Scaled expression', expr_box[0]*72, 51, 6., role='legend')
        for fraction, value in [(0., low), (1., high)]:
            text(c, f'{value:.2f}', (expr_box[0]+expr_box[2]*fraction)*72, expr_box[1]*72-7,
                 5.4, align='center', role='tick')
        pseudo_cmap = item.get('pseudotime_cmap_rgb')
        if pseudo_cmap is None:
            from matplotlib import colormaps
            name = item.get('pseudotime_cmap', {'C': 'viridis', 'D': 'Reds', 'E': 'Blues', 'F': 'Greens', 'G': 'Purples'}[panel])
            pseudo_cmap = colormaps[name](np.linspace(0, 1, 256))[:, :3]
        pseudo_box = (MATRIX[0]+1.53, .565, .70, .05)
        vector_strip(c, pseudo_cmap, pseudo_box, frame=False)
        text(c, 'Original path distance', pseudo_box[0]*72, 51, 6., role='legend')
        bounds = item.get('pt_limits')
        if bounds is None:
            column = 'path_t' if 'path_t' in cells else 't'
            data = cells[column].to_numpy(float)
            bounds = [float(np.percentile(data, 2)), float(data.max())]
        for fraction, value in [(0., bounds[0]), (1., bounds[1])]:
            text(c, f'{float(value):.2f}', (pseudo_box[0]+pseudo_box[2]*fraction)*72, pseudo_box[1]*72-7,
                 5.4, align='center', role='tick')
        text(c, 'Age', 14, 21, 5.8, font='Arial-Bold', role='legend')
        for j, stage in enumerate(present):
            x = 39+j*35
            circle(c, x, 22, 1.8, STAGE_COLORS[STAGES.index(stage)])
            text(c, stage, x+4.5, 20.5, 5.7, role='legend')
        text(c, 'Tree: accepted log1p display. Heatmap: original fitted expression and cell ordering.', 14, 7, 5.5,
             fill='#555555', role='note')

        for entry in c._scene:
            x0, y0, x1, y1 = entry['bbox']
            assert min(x0, y0) >= -.1 and x1 <= width+.1 and y1 <= height+.1, ('Clipped text', panel, entry)
        gene_boxes = [entry for entry in c._scene if entry['role'] == 'gene']
        for i, a in enumerate(gene_boxes):
            for b in gene_boxes[i+1:]:
                x0, y0, x1, y1 = a['bbox']; p0, q0, p1, q1 = b['bbox']
                assert x1 <= p0 or p1 <= x0 or y1 <= q0 or q1 <= y0, ('Overlapping genes', panel, a, b)
        scenes[panel] = c._scene
        c.showPage(); c.save()
        subprocess.run([str(pdftoppm), '-singlefile', '-r', str(DPI), '-png', str(pdf), str(png.with_suffix(''))],
                       check=True, capture_output=True)
        with Image.open(png) as rendered:
            assert all(abs(px-v*DPI) <= 1.01 for px, v in zip(rendered.size, SIZE))
            pixels, actual_dpi = list(rendered.size), list(rendered.info['dpi'])
        artifacts[panel] = dict(pdf=str(pdf), png=str(png), inches=list(SIZE), pixels=pixels, dpi=actual_dpi,
                                pdf_sha256=hashlib.sha256(pdf.read_bytes()).hexdigest(),
                                png_sha256=hashlib.sha256(png.read_bytes()).hexdigest(),
                                expected_images=0, expected_clips=0, expected_forms=0, heatmap_vector=vector_report, genes=n_genes,
                                cells=n_cells, gene_labels=[x[1] for x in label_rows])
        mapped = np.column_stack(tree_map(xy[:, 0], xy[:, 1]))
        nodes = np.column_stack(graph_map(np.array([node_x[i] for i in range(len(node_y))]), node_y))
        mappings[panel] = dict(cell_xy_pt_sha256=hashlib.sha256(mapped.tobytes()).hexdigest(),
                               node_xy_pt_sha256=hashlib.sha256(nodes.tobytes()).hexdigest(),
                               selected_ids_sha256=hashlib.sha256('\n'.join(cell_ids).encode()).hexdigest(),
                               tree_axes_inches=[TREE_LEFT, MATRIX[1], TREE_WIDTH, TREE_HEIGHT],
                               x_bounds=[XMIN, XMAX], y_bounds=[ymin, ymax],
                               uniform_scale_from_AB=TREE_SCALE, point_diameter_pt=TREE_POINT_DIAMETER)
    assert len({v['cell_xy_pt_sha256'] for v in mappings.values()}) == 1
    assert len({v['node_xy_pt_sha256'] for v in mappings.values()}) == 1
    assert hashlib.sha256(xy.tobytes()).hexdigest() == original_xy_sha
    return dict(artifacts=artifacts, scenes=scenes, image_bbox_by_panel=image_boxes,
                coordinate_mappings=mappings, cleaned_gene_labels=stripped_symbols,
                render_spec=dict(size_inches=SIZE, matrix_inches=list(MATRIX), tree_uniform_scale=TREE_SCALE,
                                 tree_point_diameter_pt=TREE_POINT_DIAMETER, expected_clips=0,
                                 expected_images_per_page=0, expected_forms=0,
                                 image_policy='Vector heatmap RGB runs and editable trees, strips and semantic text; no image objects',
                                  heatmap_underpaint_pt=UNDERPAINT_PT,
                                 fonts=['ArialMT', 'Arial-BoldMT', 'Arial-ItalicMT']))


In [ ]:
#4. =========== Export this panel
rendered = render_composites(
    bundle, publication_display, {PANEL: heatmap}, OUTPUT_DIR, WORK
)
assert set(rendered["artifacts"]) == {PANEL}
assert rendered["artifacts"][PANEL]["genes"] == EXPECTED_GENES
assert rendered["artifacts"][PANEL]["cells"] == EXPECTED_CELLS



# Original shared C-G analysis cells
The following existing method cells retain the original path normalization,
GAMs, path comparisons, Ic endpoint edgeR tests and source checks. The same
complete source block is copied into C-G because these analyses compare paths.
Original R strings, function bodies, parameters and checks are unchanged from
the verified source. These are original source sections, not a new Run All
dispatcher. Original statistical working tables and the formal accepted figure
store retain their original separate roles.


In [ ]:
#1. =========== Fixed inputs and published analysis settings
"""Published SGN per-trajectory workflow on the accepted Fig3AB paths.

Default redraw reads the two formal result files. Historical analysis helper
definitions are retained for inspection and are not called during redraw. Statistical formulas reproduce scFates 0.4.0; named mgcv fields
restore historical list-index semantics. Gene annotations follow the user's
literature/UniProt curation in Labels.tsv rather than automatic amplitude rank.
F uses the user's revised selection: reproducible path dynamics with separately
reported E18.5 annotated-subtype pseudobulk enrichment, replacing exact-one-path.
"""
from pathlib import Path
import hashlib
import json
import os
import pickle
import subprocess
import sys
import tempfile

sys.dont_write_bytecode = True
ROOT = Path('.')
CG_WORK = ROOT  # Set this path independently; use the producer notes above.
TASK = CG_WORK  # Scratch space only; never a source of accepted analysis results.
INPUT = TASK / "input"
RSCRIPT = ROOT  # Set this path independently; use the producer notes above.
MODEL = ROOT  # Set this path independently; use the producer notes above.
PANELS = {"C":"Common early path", "D":"Ia path", "E":"Ib path", "F":"Ic path", "G":"II path"}
STAGES = ["E9.5","E10.5","E11.5","E13.5","E14.5","E15.5","E16.5","E17.5","E18.5"]
STAGE_COLORS = ["#f5b888","#f2a985","#ee9982","#e58780","#cd717e","#b55c7d","#914679","#653175","#3a1c71"]
PSEUDO_CMAPS = dict(C="viridis",D="Reds",E="Blues",F="Greens",G="Purples")
os.environ.update(MPLBACKEND="Agg", MPLCONFIGDIR=str(TASK/"cache/matplotlib"),
                  OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")


def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b""):
            h.update(block)
    return h.hexdigest()


def save_json(path, value):
    Path(path).write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False,
                                   default=lambda x:x.item() if hasattr(x,"item") else str(x)),encoding="utf-8")


def resolve_protected_source(path):
    """Recognize the user's relocation of unchanged A/B PNGs into Trees/."""
    path=Path(path)
    if not path.exists() and path.parent==ROOT and path.name.startswith("tree_"):
        moved=path.parent/"Trees"/path.name
        if moved.is_file():return moved
    return path


def verify_inputs(deep=False):
    cfg=json.loads((INPUT/"config.json").read_text(encoding="utf-8"))
    for name,expected in cfg["source_files_sha256"].items():
        if digest(resolve_protected_source(name))!=expected:raise RuntimeError(f"Protected source changed: {name}")
    for name,expected in cfg["output_files_sha256"].items():
        if name!="expression.bin" or deep:
            if digest(INPUT/name)!=expected:raise RuntimeError(f"Prepared input changed: {name}")
    assert (INPUT/"expression.bin").stat().st_size==cfg["n_genes"]*cfg["n_cells"]*8
    return cfg


In [ ]:
#2. =========== Frozen input preparation
import warnings
import h5py
import numpy as np
import pandas as pd
from scipy import sparse

PROJECT_ROOT = ROOT  # Set this path independently; use the producer notes above.
TASK_DIR = TASK
EXPECTED_CELLS = 7355
EXPECTED_GENES = 11467
EXPECTED_GLIAL = 1017
EXPECTED_CANDIDATES = 10434
EXPECTED_PANEL_CELLS = {"C": 2332, "D": 2672, "E": 1597, "F": 1130, "G": 2233}
PATH_NAMES = {"D": "Ia", "E": "Ib", "F": "Ic", "G": "II"}
NORMALIZATION_SCALE = 1_000_000.0
GENE_BLOCK_SIZE = 256


In [ ]:
#3. =========== Read-only utilities
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def array_hash(values, dtype=None):
    return hashlib.sha256(np.asarray(values, dtype=dtype).tobytes(order="C")).hexdigest()


def counts_hash(matrix):
    digest = hashlib.sha256()
    for values in (matrix.indptr, matrix.indices, matrix.data):
        digest.update(values.dtype.str.encode("ascii"))
        digest.update(values.tobytes(order="C"))
    return digest.hexdigest()


def read_vector(group):
    if isinstance(group, h5py.Dataset):
        return group.asstr()[()] if group.dtype.kind in "OS" else group[()]
    if "codes" in group:
        categories = np.asarray(read_vector(group["categories"]), dtype=object)
        codes = group["codes"][()]
        result = np.full(codes.shape, None, dtype=object)
        valid = codes >= 0
        result[valid] = categories[codes[valid]]
        return result
    if "values" in group and "mask" in group:
        result = np.asarray(read_vector(group["values"]), dtype=object)
        result[group["mask"][()].astype(bool)] = None
        return result
    raise ValueError(f"Unsupported vector encoding: {group.name}")


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


In [ ]:
#4. =========== Preserve native projection and topological segments
def fixed_tree_inputs(bundle):
    obs = bundle["obs"]
    projection = bundle["projection"]
    edges = np.asarray(projection["edges"], dtype=np.int64)
    cell_edges = np.asarray(projection["edge_id"], dtype=np.int64)
    native_t = np.asarray(projection["t"], dtype=np.float64)
    require(len(obs) == EXPECTED_CELLS and obs.index.is_unique, "Unexpected tree cell axis")
    require(np.isfinite(native_t).all(), "Non-finite native pseudotime")
    require(len(bundle["segments"]) == 7, "Expected seven topological segments")
    edge_lookup = {tuple(edge): index for index, edge in enumerate(edges)}
    require(len(edge_lookup) == len(edges), "Duplicate directed model edges")
    edge_segment = np.zeros(len(edges), dtype=np.int64)
    segment_info = []
    for number, segment in enumerate(bundle["segments"], start=1):
        nodes = [int(value) for value in segment["path"]]
        require(nodes[0] == segment["start"] and nodes[-1] == segment["end"], "Segment endpoint mismatch")
        edge_ids = [edge_lookup[(left, right)] for left, right in zip(nodes[:-1], nodes[1:])]
        require(np.all(edge_segment[edge_ids] == 0), "Overlapping topological segment edges")
        edge_segment[edge_ids] = number
        segment_info.append({"seg": f"S{number}", "start": nodes[0], "end": nodes[-1],
                             "node_ids": nodes, "edge_ids": edge_ids})
    require(np.all(edge_segment > 0), "Topological segments do not cover all model edges")
    require(np.all((cell_edges >= 0) & (cell_edges < len(edges))), "Invalid projected edge ID")
    seg = np.asarray([f"S{value}" for value in edge_segment[cell_edges]])

    # Recover every path independently from the frozen rooted parent array.
    parent = np.asarray(projection["parent"], dtype=np.int64)
    root, fork = int(projection["root"]), int(bundle["first_fork"])

    def path_edges(start, end):
        nodes = [end]
        while nodes[-1] != start:
            nodes.append(int(parent[nodes[-1]]))
            require(nodes[-1] >= 0 and len(nodes) <= len(parent), "Invalid rooted model path")
        nodes.reverse()
        return [edge_lookup[(left, right)] for left, right in zip(nodes[:-1], nodes[1:])]

    masks = {"C": np.isin(cell_edges, path_edges(root, fork))}
    require(np.array_equal(masks["C"], seg == "S1"), "Root-first-fork segment mismatch")
    mask_hashes = {}
    for panel, fate in PATH_NAMES.items():
        figure = bundle["figures"][fate]
        recovered_edges = path_edges(fork, int(figure["tip"]))
        require(set(recovered_edges) == set(map(int, figure["edge_ids"])), f"{fate} edge-list mismatch")
        mask = np.isin(cell_edges, recovered_edges)
        require(np.array_equal(mask, np.asarray(figure["mask"], dtype=bool)), f"{fate} frozen-mask mismatch")
        require(int(mask.sum()) == int(figure["cells"]), f"{fate} stored cell-count mismatch")
        masks[panel] = mask
        digest = array_hash(mask, bool)
        require(digest == figure["mask_sha256"], f"{fate} stored mask-hash mismatch")
        mask_hashes[panel] = digest
    mask_hashes["C"] = array_hash(masks["C"], bool)
    union_paths = np.logical_or.reduce([masks[panel] for panel in PATH_NAMES])
    require(np.array_equal(masks["C"], ~union_paths), "Common/path partition mismatch")
    for panel, expected in EXPECTED_PANEL_CELLS.items():
        require(int(masks[panel].sum()) == expected, f"Unexpected panel {panel} cell count")

    # Check the accepted arclength coordinate without replacing or rescaling it.
    root_distance = np.asarray(projection["root_distance"], dtype=np.float64)
    alpha = np.asarray(projection["alpha"], dtype=np.float64)
    projected_edges = edges[cell_edges]
    reconstructed_t = root_distance[projected_edges[:, 0]] + alpha * (
        root_distance[projected_edges[:, 1]] - root_distance[projected_edges[:, 0]])
    t_error = float(np.max(np.abs(reconstructed_t - native_t)))
    require(t_error < 1e-12, "Native pseudotime disagrees with frozen edge projection")
    result = pd.DataFrame({"cell_id": obs.index.astype(str), "stage": obs["stage_standardized"].astype(str).to_numpy(),
                           "study": obs["study_id"].astype(str).to_numpy(),
                           "library": obs["library_id"].astype(str).to_numpy(),
                           "state": obs["state"].astype(str).to_numpy(), "t": native_t,
                           "seg": seg, "edge_id": cell_edges})
    for panel, mask in masks.items():
        result[f"mask_{panel}"] = mask
    panel_info = {}
    for panel, mask in masks.items():
        selected = result.loc[mask]
        panel_info[panel] = {"n_cells": len(selected), "segments": sorted(selected["seg"].unique().tolist()),
                             "t_min": float(selected["t"].min()), "t_max": float(selected["t"].max()),
                             "stages": {str(k): int(v) for k, v in selected["stage"].value_counts(sort=False).items()},
                             "studies": {str(k): int(v) for k, v in selected["study"].value_counts(sort=False).items()},
                             "n_libraries": int(selected["library"].nunique()), "mask_sha256": mask_hashes[panel]}
    return result, segment_info, panel_info, t_error


In [ ]:
#5. =========== Rebuild the original-paper expression scale from counts
def prepare_inputs(project_root=PROJECT_ROOT, task_dir=TASK_DIR):
    project_root, task_dir = Path(project_root).resolve(), Path(task_dir).resolve()
    output_dir = task_dir / "input"
    bundle_path = ROOT  # Set this path independently; use the producer notes above.
    with bundle_path.open("rb") as handle, warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        bundle = pickle.load(handle)
    source_path = Path(bundle["fitted_source_h5ad"])
    png_paths = [resolve_protected_source(ROOT / f"tree_{name}.png")
                 for name in ("all", "Ia", "Ib", "Ic", "II")]
    protected_paths = [bundle_path, source_path, *png_paths]
    before_hashes = {str(path): file_hash(path) for path in protected_paths}
    require(before_hashes[str(source_path)] == bundle["fitted_source_sha256"], "RNA source hash differs from frozen tree")
    obs, segment_info, panel_info, t_error = fixed_tree_inputs(bundle)

    with h5py.File(source_path, "r") as source:
        source_cell_ids = np.asarray(read_vector(source["obs"][source["obs"].attrs["_index"]]), dtype=str)
        require(np.array_equal(source_cell_ids, obs["cell_id"].to_numpy()), "RNA/tree cell IDs or order differ")
        for source_key, output_key in (("stage_standardized", "stage"), ("study_id", "study"), ("library_id", "library")):
            require(np.array_equal(np.asarray(read_vector(source["obs"][source_key]), dtype=str), obs[output_key]),
                    f"RNA/tree {source_key} metadata differ")
        var = source["var"]
        gene_ids = np.asarray(read_vector(var[var.attrs["_index"]]), dtype=str)
        symbols = np.asarray(["" if value is None else str(value) for value in read_vector(var["gene_symbol"])])
        glial = np.asarray(read_vector(var["author_glial1371"]), dtype=bool)
        old_detected = np.asarray(read_vector(var["detected_in_any_selected_cell"]), dtype=bool)
        require(len(gene_ids) == EXPECTED_GENES and len(set(gene_ids)) == EXPECTED_GENES, "Unexpected source gene axis")
        require(int(glial.sum()) == EXPECTED_GLIAL, "Unexpected full author-glial exclusion count")
        require(not np.any(np.char.startswith(np.char.lower(symbols), "mt-")), "Unexpected mitochondrial feature")
        require(not np.any(np.char.startswith(np.char.upper(symbols), "ERCC-"))
                and not np.any(np.char.startswith(np.char.upper(symbols), "ERCC_")), "Unexpected ERCC spike-in")
        group = source["layers/counts"]
        require(group.attrs["encoding-type"] == "csr_matrix", "Counts must be CSR")
        counts = sparse.csr_matrix((group["data"][()], group["indices"][()], group["indptr"][()]),
                                   shape=tuple(group.attrs["shape"]))
        require(counts.shape == (EXPECTED_CELLS, EXPECTED_GENES), "Unexpected count dimensions")
        require(counts.has_canonical_format and np.issubdtype(counts.dtype, np.integer)
                and np.all(counts.data >= 0), "Counts must be canonical, nonnegative integers")
        raw_counts_hash = counts_hash(counts)
        nonzero = np.bincount(counts.indices[counts.data > 0], minlength=EXPECTED_GENES)
        retained = ~glial
        selected = retained & (nonzero > 0)
        require(int(retained.sum()) == 10450 and int((retained & ~selected).sum()) == 16,
                "Unexpected post-glial/all-zero filter counts")
        require(int(selected.sum()) == EXPECTED_CANDIDATES, "Unexpected candidate count")
        denominator = np.asarray(counts[:, retained].sum(axis=1, dtype=np.int64)).ravel().astype(np.float64)
        require(np.all(denominator > 0), "Zero retained-gene count total")
        expression = counts[:, selected].astype(np.float64)
        expression.data /= np.repeat(denominator, np.diff(expression.indptr))
        expression.data *= NORMALIZATION_SCALE
        expression.data = np.log10(1.0 + expression.data)
        gene_major = expression.T.tocsr()
        genes = pd.DataFrame({"gene_index": np.arange(EXPECTED_CANDIDATES, dtype=np.int64),
                              "gene_id": gene_ids[selected], "symbol": symbols[selected],
                              "global_nonzero_count": nonzero[selected]})
        audit = pd.DataFrame({"source_gene_index": np.arange(EXPECTED_GENES), "gene_id": gene_ids,
                              "symbol": symbols, "author_glial1371": glial,
                              "legacy_detected_flag": old_detected,
                              "actual_detected": nonzero > 0,
                              "legacy_detection_disagrees": old_detected != (nonzero > 0),
                              "global_nonzero_count": nonzero,
                              "decision": np.where(glial, "excluded_author_glial", np.where(nonzero == 0, "excluded_all_zero", "included"))})
        output_dir.mkdir(parents=True, exist_ok=True)
        expression_path = output_dir / "expression.bin"
        with expression_path.open("wb") as handle:
            for first in range(0, EXPECTED_CANDIDATES, GENE_BLOCK_SIZE):
                dense = gene_major[first:first + GENE_BLOCK_SIZE].toarray(order="C").astype("<f8", copy=False)
                require(np.isfinite(dense).all() and np.all(dense >= 0), "Invalid normalized expression")
                dense.tofile(handle)
        require(expression_path.stat().st_size == EXPECTED_CANDIDATES * EXPECTED_CELLS * 8,
                "Expression binary byte-size mismatch")

        # Independently re-read original HDF5 rows and validate all genes in 27 stage-stratified cells.
        sampled_cells = []
        for stage in obs["stage"].drop_duplicates():
            rows = np.flatnonzero(obs["stage"].to_numpy() == stage)
            require(len(rows) >= 3, "Insufficient cells for stage-stratified normalization checks")
            sampled_cells.extend(rows[[0, len(rows) // 2, -1]].tolist())
        require(len(sampled_cells) == len(set(sampled_cells)) == 27, "Expected 27 independent normalization checks")
        written = np.memmap(expression_path, dtype="<f8", mode="r", shape=(EXPECTED_CANDIDATES, EXPECTED_CELLS), order="C")
        max_normalization_error = 0.0
        for cell in sampled_cells:
            first, last = map(int, group["indptr"][cell:cell + 2])
            raw = np.zeros(EXPECTED_GENES, dtype=np.float64)
            raw[group["indices"][first:last]] = group["data"][first:last]
            independent_total = raw[retained].sum()
            require(independent_total == denominator[cell], "Independent count denominator mismatch")
            expected = np.log10(1.0 + NORMALIZATION_SCALE * raw[selected] / independent_total)
            difference = float(np.max(np.abs(written[:, cell] - expected)))
            max_normalization_error = max(max_normalization_error, difference)
        del written
        require(max_normalization_error < 2e-13, "Independent normalization check failed")
        require(counts_hash(counts) == raw_counts_hash, "Source counts changed in memory")

    # Verify round-trip cell order, pseudotime precision, masks, and stable gene IDs.
    obs.to_csv(output_dir / "obs.csv", index=False, float_format="%.17g")
    genes.to_csv(output_dir / "genes.csv", index=False)
    audit.to_csv(output_dir / "filter-audit.tsv", sep="\t", index=False)
    reloaded_obs = pd.read_csv(output_dir / "obs.csv", float_precision="round_trip")
    reloaded_genes = pd.read_csv(output_dir / "genes.csv", keep_default_na=False)
    require(np.array_equal(reloaded_obs["cell_id"], obs["cell_id"]), "CSV cell-order round-trip mismatch")
    require(np.array_equal(reloaded_obs["t"].to_numpy(), obs["t"].to_numpy()), "CSV native-t precision loss")
    require(np.array_equal(reloaded_obs["seg"], obs["seg"]), "CSV segment round-trip mismatch")
    require(np.array_equal(reloaded_genes["gene_id"], genes["gene_id"]), "CSV gene-order round-trip mismatch")
    for panel in EXPECTED_PANEL_CELLS:
        require(np.array_equal(reloaded_obs[f"mask_{panel}"], obs[f"mask_{panel}"]), f"CSV {panel} mask round-trip mismatch")
    after_hashes = {str(path): file_hash(path) for path in protected_paths}
    require(before_hashes == after_hashes, "An original tree, RNA, or PNG file changed during preparation")

    # Record the input contract and the unavoidable common-gene denominator adaptation.
    output_hashes = {name: file_hash(output_dir / name)
                     for name in ("expression.bin", "obs.csv", "genes.csv", "filter-audit.tsv")}
    config = {"schema_version": 1, "n_cells": EXPECTED_CELLS, "n_genes": EXPECTED_CANDIDATES,
              "expression": {"file": "expression.bin", "dtype": "float64", "endianness": "little",
                             "order": "gene-major", "shape": [EXPECTED_CANDIDATES, EXPECTED_CELLS],
                             "gene_row_bytes": EXPECTED_CELLS * 8,
                             "formula": "log10(1 + 1e6 * counts / sum(counts over 10450 non-author-glial common genes))"},
              "filter_counts": {"input_genes": EXPECTED_GENES, "excluded_author_glial": EXPECTED_GLIAL,
                                "normalization_denominator_genes": 10450, "excluded_retained_all_zero": 16,
                                "candidate_genes": EXPECTED_CANDIDATES},
              "segments": segment_info, "panels": panel_info,
              "source_files_sha256": before_hashes, "output_files_sha256": output_hashes,
              "model_hash": bundle["model_hash"], "coordinates_hash": bundle["coordinates_hash"],
              "cell_order_sha256": hashlib.sha256("\n".join(obs["cell_id"]).encode("utf-8")).hexdigest(),
              "native_t_sha256": array_hash(obs["t"].to_numpy(), "<f8"),
              "validation": {"all_protected_file_hashes_unchanged": True,
                             "raw_counts_csr_arrays_sha256": raw_counts_hash, "raw_counts_unchanged": True,
                             "cell_order_exact": True, "segment_mapping_exact": True, "path_masks_exact": True,
                             "native_t_roundtrip_exact": True, "native_projection_max_abs_error": t_error,
                             "normalization_sample_n": len(sampled_cells),
                             "normalization_sample_cell_ids": obs.loc[sampled_cells, "cell_id"].tolist(),
                             "normalization_max_abs_error": max_normalization_error,
                             "legacy_detected_flag_disagreement_all_genes": int(np.sum(old_detected != (nonzero > 0))),
                             "legacy_detected_flag_disagreement_non_glial": int(np.sum(retained & (old_detected != (nonzero > 0)))),
                             "non_glial_legacy_false_actual_positive": int(np.sum(retained & ~old_detected & (nonzero > 0))),
                             "non_glial_legacy_true_actual_zero": int(np.sum(retained & old_detected & (nonzero == 0)))},
              "limitations": ["The denominator uses the retained common gene axis, not each original source's full gene universe.",
                              "Shared cells on paired fate paths are reused, not independent observations.",
                              "Original counts and RNA X, native tree, annotations, projection, and PNG files are unchanged."]}
    with (output_dir / "config.json").open("w", encoding="utf-8") as handle:
        json.dump(config, handle, ensure_ascii=False, indent=2, allow_nan=False)
        handle.write("\n")
    print(f"Prepared {EXPECTED_CANDIDATES} genes x {EXPECTED_CELLS} cells; panels C-G: "
          + ", ".join(str(EXPECTED_PANEL_CELLS[panel]) for panel in EXPECTED_PANEL_CELLS))
    print(f"Excluded {EXPECTED_GLIAL} author-glial and 16 remaining all-zero genes; "
          f"27 normalization checks max error={max_normalization_error:.3g}; original file hashes unchanged.")
    return config


In [ ]:
#6. =========== Immutable R snapshot with the published formulas
R_SOURCE = r'''
suppressPackageStartupMessages({library(mgcv);library(jsonlite);library(parallel)})
args <- commandArgs(trailingOnly=TRUE)
task <- normalizePath(args[1],winslash="/",mustWork=TRUE)
panel <- args[2]
mode <- args[3]
workers <- as.integer(args[4])
origin <- as.numeric(args[5])
input <- normalizePath(args[6],winslash="/",mustWork=TRUE)
out <- file.path(task,panel)
dir.create(out,recursive=TRUE,showWarnings=FALSE)
cfg <- fromJSON(file.path(input,"config.json"),simplifyVector=TRUE)
all_obs <- read.csv(file.path(input,"obs.csv"),check.names=FALSE)
genes <- read.csv(file.path(input,"genes.csv"),check.names=FALSE)
rows <- which(as.logical(all_obs[[paste0("mask_",panel)]]))
obs <- all_obs[rows,,drop=FALSE]
obs$path_t <- obs$t-origin
stopifnot(nrow(genes)==cfg$n_genes,nrow(all_obs)==cfg$n_cells)
# Subsetting a single root-to-tip route collapses internal degree-two nodes.
# All five extracted paths therefore have one segment and one branch, as in
# subset_tree -> root -> pseudotime in the published scFates 0.4.0 code.
dat <- data.frame(t=obs$path_t,seg="1",exp=rep(0,nrow(obs)))
ACUT <- if(panel=="C") 1 else .3
PCUT <- 1e-4
GAMMA <- 1.5
NC <- cfg$n_cells
expression_path <- file.path(input,"expression.bin")

read_y <- function(g) {
    seek(expression_con,where=(g-1)*NC*8,origin="start")
    readBin(expression_con,what="double",n=NC,size=8,endian="little")[rows]
}

make_templates <- function() {
    # Reuse only response-independent basis construction, not fitted smoothness.
    list(test=gam(exp~s(t,k=5),data=dat,fit=FALSE),
         null=gam(exp~1,data=dat,fit=FALSE),
         final=gam(exp~s(t,bs="ts"),data=dat,fit=FALSE))
}

fit_template <- function(G,y,gamma=1) {
    G$y <- y
    G$model$exp <- y
    gam(G=G,gamma=gamma)
}

statistics_from_models <- function(m,m0) {
    # Preserve the author's extra -n_segments, F statistic and tail exactly.
    D <- as.numeric(m$deviance)
    odf <- as.numeric(m$df.residual)-1
    D0 <- as.numeric(m0$deviance)
    df0 <- as.numeric(m0$df.residual)
    Fstat <- if(D==0) 0 else (D0-D)/(df0-odf)/(D/odf)
    p <- pf(Fstat,df0-odf,odf,lower.tail=FALSE)
    pred <- as.numeric(predict(m))
    c(p_value=p,A=max(pred)-min(pred),Fstat=Fstat,df_num=df0-odf,
      df_den=odf,deviance=D,null_deviance=D0,edf=sum(m$edf))
}

test_one <- function(g) {
    warns <- character()
    value <- tryCatch(withCallingHandlers({
        y <- read_y(g)
        m <- fit_template(templates$test,y)
        m0 <- fit_template(templates$null,y)
        c(as.list(statistics_from_models(m,m0)),list(status="ok",error=""))
    },warning=function(w){warns<<-c(warns,conditionMessage(w));invokeRestart("muffleWarning")}),
    error=function(e)c(as.list(setNames(rep(NA_real_,8),c("p_value","A","Fstat","df_num","df_den","deviance","null_deviance","edf"))),
                      list(status="error",error=conditionMessage(e))))
    value$warnings <- paste(unique(warns),collapse=" | ")
    value
}

init_worker <- function() {
    suppressPackageStartupMessages(library(mgcv))
    assign("expression_con",file(expression_path,"rb"),envir=.GlobalEnv)
    assign("templates",make_templates(),envir=.GlobalEnv)
    NULL
}

expression_con <- file(expression_path,"rb")
on.exit <- NULL
templates <- make_templates()

if(mode=="pilot") {
    set.seed(42)
    ng <- nrow(genes)
    detection_order <- order(genes$global_nonzero_count)
    picked <- unique(c(detection_order[seq(1,ng,length.out=16)],sample.int(ng,32)))
    reference <- list(); cached <- list(); final_diff <- numeric()
    start <- proc.time()[3]
    for(g in picked) {
        y <- read_y(g); ds <- dat; ds$exp <- y
        m <- gam(exp~s(t,k=5),data=ds)
        m0 <- gam(exp~1,data=ds)
        reference[[length(reference)+1]] <- statistics_from_models(m,m0)
    }
    direct_seconds <- proc.time()[3]-start
    start <- proc.time()[3]
    for(g in picked) cached[[length(cached)+1]] <- unlist(test_one(g)[names(reference[[1]])])
    cached_seconds <- proc.time()[3]-start
    for(g in head(picked,12)) {
        y<-read_y(g);ds<-dat;ds$exp<-y
        a<-gam(exp~s(t,bs="ts"),data=ds,gamma=GAMMA)
        b<-fit_template(templates$final,y,gamma=GAMMA)
        final_diff<-c(final_diff,max(abs(predict(a)-predict(b))))
    }
    aa<-do.call(rbind,reference);bb<-do.call(rbind,cached)
    delta<-abs(aa-bb)
    relative<-delta/pmax(1,abs(aa))
    stopifnot(max(relative,na.rm=TRUE)<1e-10,max(final_diff)<1e-10)
    report<-list(panel=panel,genes=length(picked),gene_ids=genes$gene_id[picked],
                 max_relative_stat_difference=max(relative,na.rm=TRUE),max_final_prediction_difference=max(final_diff),
                 direct_seconds=unname(direct_seconds),cached_seconds=unname(cached_seconds),
                 mgcv=as.character(packageVersion("mgcv")),R=R.version.string,
                 legacy_df_field="df.residual; historical R slot43, modern slot44; never offset",
                 single_subset_segment=TRUE)
    write_json(report,file.path(out,"equivalence.json"),pretty=TRUE,auto_unbox=TRUE)
    cat(toJSON(report,auto_unbox=TRUE),"\n")
    close(expression_con)
    quit(save="no")
}

if(!mode%in%c("analyze","refit")) stop("Unknown mode")
if(!file.exists(file.path(out,"equivalence.json"))) stop("Run pilot equivalence before production")
cl <- makePSOCKcluster(workers,outfile="")
clusterExport(cl,c("NC","rows","dat","expression_path","make_templates","fit_template",
                   "statistics_from_models","read_y","test_one","init_worker"),envir=.GlobalEnv)
invisible(clusterEvalQ(cl,{suppressPackageStartupMessages(library(mgcv));NULL}))
invisible(clusterCall(cl,init_worker))
start <- proc.time()[3]
if(mode=="analyze") {
results <- vector("list",nrow(genes))
for(begin in seq(1,nrow(genes),by=400)) {
    ix <- begin:min(nrow(genes),begin+399)
    got <- parLapply(cl,ix,test_one)
    results[ix] <- got
    cat(panel," tested ",max(ix),"/",nrow(genes),"\n",sep="");flush.console()
}
stats <- cbind(genes,do.call(rbind,lapply(results,as.data.frame,stringsAsFactors=FALSE)))
# Bonferroni uses all tested candidates, including undefined entries.
stats$adjusted_p <- pmin(stats$p_value*nrow(genes),1)
stats$st <- as.integer(is.finite(stats$adjusted_p)&is.finite(stats$A)&stats$adjusted_p<PCUT&stats$A>ACUT)
stats$signi <- stats$st>.8
write.table(stats,file.path(out,"stats.tsv"),sep="\t",quote=FALSE,row.names=FALSE,na="NA")
} else {
    stats<-read.delim(file.path(out,"stats.tsv"),check.names=FALSE)
}
if(any(stats$status!="ok")) {
    stopCluster(cl)
    stop("Association errors were recorded. Do not continue silently or invent p-values.")
}
selected <- which(stats$signi)
if(!length(selected)) {
    stopCluster(cl)
    stop("No genes pass the published criteria; do not relax thresholds.")
}
fitted_path <- file.path(out,"fitted.bin")
fit_con <- file(fitted_path,"wb")
fit_one <- function(g) {
    warns<-character()
    ans<-withCallingHandlers({
        y<-read_y(g)
        m<-fit_template(templates$final,y,gamma=1.5)
        v<-as.numeric(predict(m))
        stopifnot(length(v)==nrow(dat),all(is.finite(v)))
        converged<-if(!is.null(m$converged)) isTRUE(m$converged) else isTRUE(m$mgcv.conv$fully.converged)
        list(values=v,final_edf=sum(m$edf),final_deviance=m$deviance,
             fitted_min=min(v),fitted_max=max(v),converged=converged)
    },warning=function(w){warns<<-c(warns,conditionMessage(w));invokeRestart("muffleWarning")})
    ans$final_warnings<-paste(unique(warns),collapse=" | ");ans
}
clusterExport(cl,"fit_one",envir=.GlobalEnv)
fit_stats <- vector("list",length(selected))
for(begin in seq(1,length(selected),by=200)) {
    ix<-begin:min(length(selected),begin+199)
    got<-parLapply(cl,selected[ix],fit_one)
    for(j in seq_along(ix)) {
        z<-got[[j]]
        writeBin(z$values,fit_con,size=8,endian="little")
        z$values<-NULL;fit_stats[[ix[j]]]<-as.data.frame(z,stringsAsFactors=FALSE)
    }
    cat(panel," fitted ",max(ix),"/",length(selected),"\n",sep="");flush.console()
}
close(fit_con)
stopCluster(cl)
close(expression_con)
metadata<-cbind(stats[selected,,drop=FALSE],do.call(rbind,fit_stats))
write.table(metadata,file.path(out,"fitted-genes.tsv"),sep="\t",quote=FALSE,row.names=FALSE,na="NA")
write.csv(obs,file.path(out,"cells.csv"),row.names=FALSE)
report<-list(panel=panel,cells=nrow(obs),genes_tested=nrow(genes),significant=length(selected),
             A_cut=ACUT,adjusted_p_cut=PCUT,multiple_testing="Bonferroni N includes all candidates",
             n_map=1,st_cut=.8,test_formula="exp~s(t,k=5)",test_segments=1,
             final_formula="exp~s(t,bs='ts')",gamma=1.5,method="GCV.Cp",family="Gaussian identity",
             study_covariates=FALSE,library_covariates=FALSE,workers=workers,
             elapsed_seconds=unname(proc.time()[3]-start),mgcv=as.character(packageVersion("mgcv")),R=R.version.string,
             association_errors=sum(stats$status!="ok"),nonfinite_p=sum(!is.finite(stats$p_value)),
             legacy_df_field="Named df.residual replaces historical mgcv list slot 43",path_origin=origin,
             fitted_binary="gene-major little-endian float64, original cells.csv order")
if(mode=="refit") {
    previous<-fromJSON(file.path(out,"analysis.json"))
    report$association_elapsed_in_original_run<-previous$elapsed_seconds
    report$operation<-"refit unchanged significant genes; corrected diagnostic field names"
}
write_json(report,file.path(out,"analysis.json"),pretty=TRUE,auto_unbox=TRUE)
cat(toJSON(report,auto_unbox=TRUE),"\n")
'''


def run_r(panel, mode, workers):
    verify_inputs(deep=mode=="analyze")
    cache = TASK/"cache/runtime"
    cache.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=f"{panel}-{mode}-",dir=cache) as directory:
        work = Path(directory)
        script = ROOT  # Set this path independently; use the producer notes above.
        script.write_text(R_SOURCE,encoding="utf-8")
        env = os.environ.copy()
        env.update(LC_ALL="C",TMP=str(work),TEMP=str(work),TMPDIR=str(work))
        with MODEL.open("rb") as stream:
            bundle=pickle.load(stream)
        origin=0. if panel=="C" else float(bundle["projection"]["root_distance"][bundle["first_fork"]])
        subprocess.run([str(RSCRIPT),"--vanilla",str(script),str(TASK),panel,mode,str(workers),repr(origin),str(INPUT)],
                       env=env,cwd=ROOT,check=True)


In [ ]:
#7. =========== Four-path exclusive gene selection
# F is now explicitly exempt from this rule at the user's request. The legacy
# four-path table remains available to explain the original 14-gene result.
def ic_markers():
    """Discover E18.5 Ic markers from annotated, mutually exclusive subtypes."""
    # These criteria are fixed before examining the number of passing genes.
    FDR_CUT, LOG2FC_CUT, DETECTION_CUT = .05, .5, .10
    cfg = verify_inputs(deep=True)
    out = TASK/"F"
    obs = pd.read_csv(INPUT/"obs.csv")
    genes = pd.read_csv(INPUT/"genes.csv")
    states = {f"Immature Type {x} SGN": x for x in ("Ia", "Ib", "Ic", "II")}
    source_path = ROOT  # Set this path independently; use the producer notes above.
    with h5py.File(source_path,"r") as source:
        source_ids = np.asarray(read_vector(source["obs"][source["obs"].attrs["_index"]]),dtype=str)
        assert np.array_equal(source_ids,obs.cell_id)
        current = np.asarray(read_vector(source["obs/curated_cell_type"]),dtype=str)
        assert np.array_equal(current,obs.state)
        biological = np.asarray(read_vector(source["obs/biological_replicate_id"]),dtype=str)
        source_genes = np.asarray(read_vector(source["var"][source["var"].attrs["_index"]]),dtype=str)
        indices = pd.Index(source_genes).get_indexer(genes.gene_id)
        assert (indices>=0).all()
        group = source["layers/counts"]
        counts = sparse.csr_matrix((group["data"][()],group["indices"][()],group["indptr"][()]),shape=tuple(group.attrs["shape"]))[:,indices]
    selected = (obs.stage=="E18.5") & obs.state.isin(states)
    annotated = obs.loc[selected].copy()
    annotated["subtype"] = annotated.state.map(states)
    annotated["biological_replicate"] = biological[selected]
    annotated["source_row"] = np.flatnonzero(selected)
    records, sums, detections = [], [], []
    for (study,library,subtype), frame in annotated.groupby(["study","library","subtype"],sort=True):
        ix = frame.source_row.to_numpy()
        block = counts[ix]
        sample = f"{library}__{subtype}"
        bio = frame.biological_replicate.unique()
        assert len(bio)==1 and len(ix)>=5
        sums.append(np.asarray(block.sum(axis=0,dtype=np.int64)).ravel())
        detections.append(np.asarray((block>0).sum(axis=0)).ravel()/len(ix))
        records.append(dict(sample=sample,study=study,library=library,subtype=subtype,
                            biological_replicate=bio[0],cells=len(ix),total_counts=int(sums[-1].sum())))
    meta = pd.DataFrame(records)
    assert len(meta)==20 and (meta.groupby("library").subtype.nunique()==4).all()
    assert meta.loc[meta.study=="GSE195500","biological_replicate"].nunique()==3
    genes.join(pd.DataFrame(np.column_stack(sums),columns=meta["sample"])).to_csv(out/"pseudobulk-counts.tsv",sep="\t",index=False)
    pd.DataFrame(np.column_stack(detections),columns=meta["sample"]).assign(gene_id=genes.gene_id).to_csv(out/"pseudobulk-detection.tsv",sep="\t",index=False)
    meta.to_csv(out/"pseudobulk-samples.tsv",sep="\t",index=False)
    r_source = r'''
suppressPackageStartupMessages({library(edgeR);library(jsonlite)})
args<-commandArgs(trailingOnly=TRUE)
out<-args[1];qcut<-as.numeric(args[2]);lfccut<-as.numeric(args[3]);detcut<-as.numeric(args[4])
raw<-read.delim(file.path(out,"pseudobulk-counts.tsv"),check.names=FALSE)
meta<-read.delim(file.path(out,"pseudobulk-samples.tsv"),check.names=FALSE)
det<-read.delim(file.path(out,"pseudobulk-detection.tsv"),check.names=FALSE)
gcols<-c("gene_index","gene_id","symbol","global_nonzero_count")
counts<-as.matrix(raw[,meta$sample]);rownames(counts)<-raw$gene_id
stopifnot(identical(as.character(det$gene_id),as.character(raw$gene_id)))
primary<-meta$study=="GSE195500"
d<-meta[primary,];d$subtype<-factor(d$subtype,levels=c("Ic","Ib","Ia","II"));d$pool<-factor(d$biological_replicate)
design<-model.matrix(~pool+subtype,data=d)
stopifnot(qr(design)$rank==ncol(design),nrow(d)==12,length(levels(d$pool))==3)
y0<-DGEList(counts=counts[,primary],samples=d)
keep<-filterByExpr(y0,group=d$subtype)
y<-y0[keep,,keep.lib.sizes=FALSE]
y<-calcNormFactors(y,method="TMM")
y<-estimateDisp(y,design,robust=TRUE)
fit<-glmQLFit(y,design,robust=TRUE)
res<-raw[,gcols];res$tested<-keep
pcols<-character();fc_cols<-character()
for(other in c("Ib","Ia","II")) {
    contrast<-rep(0,ncol(design));contrast[match(paste0("subtype",other),colnames(design))]<- -1
    tt<-glmQLFTest(fit,contrast=contrast)$table
    prefix<-paste0("Ic_vs_",other)
    res[[paste0(prefix,"_log2FC")]]<-NA_real_
    res[[paste0(prefix,"_P")]]<-1
    res[[paste0(prefix,"_FDR")]]<-1
    res[keep,paste0(prefix,"_log2FC")]<-tt$logFC
    res[keep,paste0(prefix,"_P")]<-tt$PValue
    res[keep,paste0(prefix,"_FDR")]<-p.adjust(tt$PValue,"BH")
    pcols<-c(pcols,paste0(prefix,"_P"));fc_cols<-c(fc_cols,paste0(prefix,"_log2FC"))
}
# Conjunction: reject only if Ic exceeds every other subtype. Positive-direction
# two-sided P values are conservative; max(P) is adjusted once over all tested genes.
pdir<-as.matrix(res[,pcols]);fc<-as.matrix(res[,fc_cols]);pdir[!is.finite(fc)|fc<=0]<-1
res$Ic_all_P_IUT<-apply(pdir,1,max)
res$Ic_all_FDR_IUT<-1
res$Ic_all_FDR_IUT[keep]<-p.adjust(res$Ic_all_P_IUT[keep],"BH")
res$Ic_all_min_log2FC<-apply(fc,1,min,na.rm=FALSE)
for(type in c("Ic","Ib","Ia","II")) {
    ix<-which(primary&meta$subtype==type)
    res[[paste0(type,"_10x_detected_fraction")]]<-as.numeric(as.matrix(det[,meta$sample[ix]])%*%meta$cells[ix]/sum(meta$cells[ix]))
    res[[paste0(type,"_10x_pools_detected_10pct")]]<-rowSums(as.matrix(det[,meta$sample[ix]])>=detcut)
}
# Normalize sources separately. The SS2 plates have unverified biological
# independence and contribute descriptive paired directions, never the main P.
for(study in unique(meta$study)) {
    ix<-which(meta$study==study)
    yd<-DGEList(counts=counts[,ix]);yd<-calcNormFactors(yd,method="TMM")
    lcpm<-cpm(yd,log=TRUE,prior.count=2)
    libs<-unique(meta$library[ix])
    for(other in c("Ib","Ia","II")) {
        diffs<-sapply(libs,function(lib) {
            ii<-which(meta$library[ix]==lib&meta$subtype[ix]=="Ic")
            jj<-which(meta$library[ix]==lib&meta$subtype[ix]==other)
            stopifnot(length(ii)==1,length(jj)==1)
            lcpm[,ii]-lcpm[,jj]
        })
        prefix<-paste0(study,"_Ic_vs_",other)
        res[[paste0(prefix,"_mean_log2CPM_difference")]]<-rowMeans(diffs)
        res[[paste0(prefix,"_positive_pairs")]]<-rowSums(diffs>0)
        for(j in seq_along(libs)) res[[paste0(prefix,"_",libs[j])]]<-diffs[,j]
    }
}
detect<-res$Ic_10x_detected_fraction>=detcut
res$Ic_vs_Ib_DE<-keep&res$Ic_vs_Ib_FDR<qcut&res$Ic_vs_Ib_log2FC>=lfccut&detect
res$Ic_all_enriched<-keep&res$Ic_all_FDR_IUT<qcut&res$Ic_all_min_log2FC>=lfccut&detect
res$Ic_vs_Ib_source_supported<-res$Ic_vs_Ib_DE&res$GSE165502_Ic_vs_Ib_mean_log2CPM_difference>0
res$Ic_all_source_supported<-res$Ic_all_enriched&res$GSE165502_Ic_vs_Ib_mean_log2CPM_difference>0&res$GSE165502_Ic_vs_Ia_mean_log2CPM_difference>0&res$GSE165502_Ic_vs_II_mean_log2CPM_difference>0
write.table(res,file.path(out,"differential.tsv"),sep="\t",quote=FALSE,row.names=FALSE,na="NA")
report<-list(stage="E18.5",primary_study="GSE195500",biological_replicates=3,replicate_unit="independent pool, each five embryos; cells are not biological replicates",primary_cells=sum(d$cells),cells_by_subtype=as.list(tapply(d$cells,d$subtype,sum)),design=colnames(design),residual_df=nrow(design)-ncol(design),genes_available=nrow(raw),genes_tested=sum(keep),filter="edgeR filterByExpr(group=subtype), defaults",normalization="TMM, sources separately",model="edgeR robust empirical-Bayes negative-binomial quasi-likelihood; ~pool+subtype",effect_test="two-sided difference from zero plus estimated positive log2FC threshold; not a test that true effect exceeds threshold",fdr_cut=qcut,log2FC_cut=lfccut,Ic_detection_cut=detcut,conjunction="directional max of conservative two-sided P, then BH across tested genes",validation="GSE165502: two matched SS2 plates; descriptive mean paired log2CPM difference >0; no biological-replicate P claimed",counts=list(Ic_vs_Ib=sum(res$Ic_vs_Ib_DE),Ic_vs_Ib_source_supported=sum(res$Ic_vs_Ib_source_supported),Ic_all=sum(res$Ic_all_enriched),Ic_all_source_supported=sum(res$Ic_all_source_supported)),edgeR=as.character(packageVersion("edgeR")),limma=as.character(packageVersion("limma")),R=R.version.string,sources=c("https://bioconductor.org/books/3.21/OSCA.multisample/multi-sample-comparisons.html","https://bioconductor.org/packages/release/bioc/vignettes/edgeR/inst/doc/edgeRUsersGuide.pdf","https://pmc.ncbi.nlm.nih.gov/articles/PMC9860252/"))
write_json(report,file.path(out,"differential.json"),pretty=TRUE,auto_unbox=TRUE)
cat(toJSON(report,auto_unbox=TRUE),"\n")
'''
    cache = TASK/"cache/runtime"
    cache.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="F-markers-",dir=cache) as directory:
        work = Path(directory)
        script = ROOT  # Set this path independently; use the producer notes above.
        script.write_text(r_source,encoding="utf-8")
        env = os.environ.copy()
        env.update(LC_ALL="C",TMP=str(work),TEMP=str(work),TMPDIR=str(work))
        subprocess.run([str(RSCRIPT),"--vanilla",str(script),str(out),str(FDR_CUT),str(LOG2FC_CUT),str(DETECTION_CUT)],env=env,cwd=ROOT,check=True)


def select():
    tables = {p:pd.read_csv(TASK/p/"stats.tsv",sep="\t") for p in "DEFG"}
    ids = tables["D"].gene_id.tolist()
    assert all(x.gene_id.tolist()==ids for x in tables.values())
    flags = np.column_stack([x.signi.to_numpy(bool) for x in tables.values()])
    table = pd.DataFrame({"gene_id":ids,"symbol":tables["D"].symbol,"significant_paths":flags.sum(axis=1)})
    for i,p in enumerate("DEFG"):
        table[p] = flags[:,i]
        table[p+"_exclusive"] = flags[:,i] & (flags.sum(axis=1)==1)
    table.to_csv(TASK/"specificity.tsv",sep="\t",index=False)
    counts={p:int(table[p+"_exclusive"].sum()) for p in "DEFG"}
    save_json(TASK/"specificity.json",dict(rule="Exactly one significant path among D/E/F/G; C is excluded from this count.",genes=counts))
    print(json.dumps(counts))


def ic_dynamic_selection():
    """Keep all reproducible path dynamics; keep enrichment evidence separate."""
    # Fixed source-replication gates protect against small-amplitude color effects.
    MIN_DISPLAY = 70
    FINAL_AMP, OBSERVED_AMP, DETECTED = .3, .3, .10
    POOLED_R, SOURCE_R = .70, .50
    cfg=verify_inputs()
    out=TASK/"F"
    genes=pd.read_csv(out/"fitted-genes.tsv",sep="\t")
    obs=pd.read_csv(out/"cells.csv")
    input_obs=pd.read_csv(INPUT/"obs.csv")
    ix=np.flatnonzero(input_obs.mask_F.to_numpy(bool))
    assert obs.cell_id.tolist()==input_obs.iloc[ix].cell_id.tolist()
    raw=np.memmap(INPUT/"expression.bin",mode="r",dtype="<f8",shape=(cfg["n_genes"],cfg["n_cells"]))
    fitted=np.memmap(out/"fitted.bin",mode="r",dtype="<f8",shape=(len(genes),len(obs)))
    observed=np.asarray(raw[np.ix_(genes.gene_index.to_numpy(),ix)])
    audit=genes[["gene_index","gene_id","symbol","A","adjusted_p"]].copy()
    audit["final_amplitude"]=genes.fitted_max-genes.fitted_min

    def row_correlation(a,b):
        a=a-a.mean(axis=1,keepdims=True);b=b-b.mean(axis=1,keepdims=True)
        return (a*b).sum(axis=1)/np.maximum(np.sqrt((a*a).sum(axis=1)*(b*b).sum(axis=1)),1e-14)

    groups={"all":np.arange(len(obs))}
    groups.update({str(k):v for k,v in obs.groupby("study").indices.items()})
    for name,rows in groups.items():
        t=obs.t.to_numpy()[rows]
        edges=np.unique(np.quantile(t,np.linspace(0,1,7)))
        bins=np.searchsorted(edges[1:-1],t,side="right")
        members=[rows[bins==b] for b in np.unique(bins) if (bins==b).sum()>=5]
        assert len(members)>=3
        yy=np.column_stack([observed[:,j].mean(axis=1) for j in members])
        pp=np.column_stack([np.asarray(fitted[:,j]).mean(axis=1) for j in members])
        audit[name+"_observed_fit_r"]=row_correlation(yy,pp)
        audit[name+"_observed_amplitude"]=np.ptp(yy,axis=1)
        audit[name+"_detected_fraction"]=(observed[:,rows]>0).mean(axis=1)
        audit[name+"_observed_six_bin_means"]=[json.dumps(v.tolist()) for v in yy]
        audit[name+"_fitted_six_bin_means"]=[json.dumps(v.tolist()) for v in pp]
    de=pd.read_csv(out/"differential.tsv",sep="\t")
    audit=audit.merge(de,on=["gene_index","gene_id","symbol"],validate="one_to_one")
    audit["source_reproducible_dynamic"]=(~audit.symbol.str.match(r"^(Rpl|Rps)") &
        (audit.final_amplitude>=FINAL_AMP)&(audit.all_observed_fit_r>=POOLED_R)&
        (audit.GSE195500_observed_fit_r>=SOURCE_R)&(audit.GSE165502_observed_fit_r>=SOURCE_R)&
        (audit.GSE195500_observed_amplitude>=OBSERVED_AMP)&(audit.GSE165502_observed_amplitude>=OBSERVED_AMP)&
        (audit.GSE195500_detected_fraction>=DETECTED)&(audit.GSE165502_detected_fraction>=DETECTED))
    audit["Ic_vs_Ib_both_sources_positive"]=(audit.GSE195500_Ic_vs_Ib_mean_log2CPM_difference>0)&(audit.GSE165502_Ic_vs_Ib_mean_log2CPM_difference>0)
    audit["eligible"]=audit.source_reproducible_dynamic&audit.Ic_vs_Ib_both_sources_positive
    audit["replicated_observed_amplitude"]=np.minimum(audit.GSE195500_observed_amplitude,audit.GSE165502_observed_amplitude)
    labels=pd.read_csv(TASK/"Labels.tsv",sep="\t")
    anchors=set(labels.loc[labels.panel=="F","gene_id"])
    audit["auditory_label"]=audit.gene_id.isin(anchors)
    available=audit.loc[audit.eligible].copy()
    if not anchors.issubset(set(available.gene_id)):
        raise RuntimeError("F: curate labels within the reproducible dynamic pool before selecting rows")
    require(len(available)>=MIN_DISPLAY,"F: fewer than 70 reproducible dynamics; do not relax significance to populate figure")
    chosen=available.sort_values("gene_index",kind="stable")
    audit["displayed"]=audit.gene_id.isin(set(chosen.gene_id))
    audit.to_csv(out/"path-diagnostics.tsv",sep="\t",index=False)
    chosen.to_csv(out/"selection.tsv",sep="\t",index=False)
    report=dict(rows=len(chosen),minimum_requested=MIN_DISPLAY,original_path_dynamic=len(genes),source_reproducible_dynamic=int(audit.source_reproducible_dynamic.sum()),
                eligible=int(audit.eligible.sum()),rule="Keep every published significant path-dynamic gene passing source diagnostics and descriptive Ic>Ib direction in both sources. No ranking or truncation to force a round number.",
                gates=dict(final_amplitude=FINAL_AMP,source_observed_bin_amplitude=OBSERVED_AMP,pooled_observed_fit_bin_r=POOLED_R,source_observed_fit_bin_r=SOURCE_R,source_path_detected_fraction=DETECTED),
                excluded="Rpl/Rps symbols; no alteration to original association P values or fitted curves",
                Ic_vs_Ib_DE=int(chosen.Ic_vs_Ib_DE.sum()),Ic_vs_Ib_source_supported=int(chosen.Ic_vs_Ib_source_supported.sum()),Ic_all_enriched=int(chosen.Ic_all_source_supported.sum()),
                interpretation="Path-dynamic genes, not all statistically established Ic-specific markers; terminal enrichment and path dynamics are separate evidence axes.",
                path_caution="904/1130 cells are on the shared Ib/Ic segment. Existing annotation, cell order, tree and native t are unchanged.")
    save_json(out/"selection.json",report)
    print(json.dumps(report))


In [ ]:
#8. =========== Heatmap and literature-supported gene annotations
def selected_labels(panel, genes):
    """Annotate existing heatmap rows using the user-authorized evidence table."""
    spec = CG_LABELS.copy()
    required = {"panel", "gene_id", "symbol", "evidence_kind", "uniprot_url", "primary_urls", "evidence_limits"}
    if not required.issubset(spec.columns):
        raise RuntimeError("The curated label evidence table is incomplete.")
    chosen = spec.loc[spec.panel == panel].copy()
    if chosen.gene_id.duplicated().any() or chosen.symbol.duplicated().any():
        raise RuntimeError(f"{panel}: duplicate curated gene labels")
    if (panel != "F" and len(chosen) != 10) or not 1 <= len(chosen) <= 10:
        raise RuntimeError(f"{panel}: invalid evidence-supported label count")
    if not set(chosen.evidence_kind).issubset({"trend_supported", "known_auditory", "candidate"}):
        raise RuntimeError(f"{panel}: unknown label evidence category")
    if not set(chosen.gene_id).issubset(set(genes.gene_id)):
        raise RuntimeError(f"{panel}: a requested label is outside the fixed heatmap gene pool")
    expected = genes.set_index("gene_id").symbol.reindex(chosen.gene_id).to_numpy()
    if not np.array_equal(expected, chosen.symbol.to_numpy()):
        raise RuntimeError(f"{panel}: curated gene identity mismatch")
    if not chosen.primary_urls.str.contains("https://", regex=False).all():
        raise RuntimeError(f"{panel}: a label lacks a primary evidence URL")
    metadata = chosen[["gene_id", "evidence_kind", "uniprot_url", "primary_urls", "evidence_limits"]]
    return genes.merge(metadata, on="gene_id", how="inner", validate="one_to_one").sort_values("display_row")


In [ ]:
#9. =========== Observed-expression diagnostics without changing selection
def review(panel):
    cfg=verify_inputs()
    out=TASK/panel
    displayed=pd.read_csv(out/"displayed-genes.tsv",sep="\t")
    all_genes=pd.read_csv(out/"fitted-genes.tsv",sep="\t")
    obs=pd.read_csv(out/"cells.csv")
    input_obs=pd.read_csv(INPUT/"obs.csv")
    input_rows=np.flatnonzero(input_obs[f"mask_{panel}"].to_numpy(bool))
    assert obs.cell_id.tolist()==input_obs.iloc[input_rows].cell_id.tolist()
    raw=np.memmap(INPUT/"expression.bin",dtype="<f8",mode="r",shape=(cfg["n_genes"],cfg["n_cells"]))
    fitted=np.memmap(out/"fitted.bin",dtype="<f8",mode="r",shape=(len(all_genes),len(obs)))
    assert (out/"fitted.bin").stat().st_size==len(all_genes)*len(obs)*8
    labels=selected_labels(panel,displayed)
    sampled=displayed.sample(min(20,len(displayed)),random_state=42)
    picked=pd.concat([labels,sampled]).drop_duplicates("gene_id")
    fit_indices=pd.Index(all_genes.gene_id).get_indexer(picked.gene_id)
    groups={"all":np.arange(len(obs))}
    groups.update({str(key):np.asarray(ix) for key,ix in obs.groupby("study").indices.items()})
    records=[]
    for (_,gene),fit_index in zip(picked.iterrows(),fit_indices):
        observed=np.asarray(raw[int(gene.gene_index),input_rows])
        pred=np.asarray(fitted[fit_index])
        for name,ix in groups.items():
            if len(ix)<30:continue
            tt=obs.t.to_numpy()[ix]
            edges=np.unique(np.quantile(tt,np.linspace(0,1,7)))
            bins=np.searchsorted(edges[1:-1],tt,side="right")
            yy=[];pp=[]
            for b in np.unique(bins):
                members=ix[bins==b]
                if len(members)<5:continue
                yy.append(float(observed[members].mean()));pp.append(float(pred[members].mean()))
            corr=float(np.corrcoef(yy,pp)[0,1]) if len(yy)>=3 and np.std(yy)>0 and np.std(pp)>0 else None
            records.append(dict(gene_id=gene.gene_id,symbol=gene.symbol,group=name,cells=len(ix),bins=len(yy),
                                observed_fitted_bin_correlation=corr,observed_bin_means=json.dumps(yy),
                                fitted_bin_means=json.dumps(pp),is_label=gene.gene_id in set(labels.gene_id)))
    checks=pd.DataFrame(records)
    checks.to_csv(out/"observed-check.tsv",sep="\t",index=False)
    qa=json.loads((out/"figure-qa.json").read_text(encoding="utf-8"))
    amp=all_genes.fitted_max-all_genes.fitted_min
    qa["final_fit_diagnostics"]={"all_converged":bool(all_genes.converged.all()),
                                 "warnings":int(all_genes.final_warnings.notna().sum()),
                                 "minimum_fitted_amplitude":float(amp.min()),
                                 "median_fitted_amplitude":float(amp.median()),
                                 "minimum_final_to_test_amplitude_ratio":float((amp/all_genes.A).min())}
    qa["observed_checks"]={"genes":len(picked),"label_genes":len(labels),
                           "selection_unchanged":True,"rule":"Six t-quantile bins, keep tied t together; >=5 cells per mean; sources >=30 cells. Descriptive only, no selection gate."}
    if panel=="F" and (out/"selection.json").exists():
        qa["gene_selection"]=json.loads((out/"selection.json").read_text(encoding="utf-8"))
        qa["observed_checks"]["all_displayed_genes_checked"]="All displayed genes passed source-level six-bin checks in path-diagnostics.tsv before selection; this table independently repeats labels plus a seeded sample."
        evidence=pd.read_csv(TASK/"Labels.tsv",sep="\t",keep_default_na=False)
        fit_lookup=all_genes.set_index("gene_id")
        for _,label in labels.iterrows():
            index=evidence.index[(evidence.panel=="F")&(evidence.gene_id==label.gene_id)]
            assert len(index)==1
            index=index[0]
            check=checks.loc[checks.gene_id==label.gene_id].set_index("group")
            record=fit_lookup.loc[label.gene_id]
            obs_y=np.asarray(raw[int(record.gene_index),input_rows])
            details=[]
            for group_name,z in check.iterrows():
                if group_name=="all":continue
                yy=json.loads(z.observed_bin_means);pp=json.loads(z.fitted_bin_means)
                details.append(dict(study=group_name,cells=int(z.cells),observed_fit_bin_r=float(z.observed_fitted_bin_correlation),observed_bin_means=yy,fitted_bin_means=pp))
            pooled=check.loc["all"]
            assignments=dict(heatmap_row=int(label.display_row),test_A=float(record.A),Bonferroni_p=float(record.adjusted_p),
                             final_fitted_amplitude=float(record.fitted_max-record.fitted_min),path_positive_fraction=float((obs_y>0).mean()),
                             pooled_observed_fit_bin_r=float(pooled.observed_fitted_bin_correlation),
                             fitted_six_bin_means=pooled.fitted_bin_means,observed_six_bin_means=pooled.observed_bin_means,
                             source_diagnostics=json.dumps(details),source_caution=False,
                             source_caution_definition="F labels pass both-source observed amplitude>=0.3, observed-fit six-bin correlation>=0.5, and path detection>=0.1; this does not exclude residual batch confounding.",
                             observed_trend="See source-specific observed and fitted six-bin values; non-monotonicity is retained.")
            for name,value in assignments.items():
                evidence.at[index,name]=value
        evidence.to_csv(TASK/"Labels.tsv",sep="\t",index=False)
    save_json(out/"figure-qa.json",qa)
    print(json.dumps(dict(panel=panel,genes_checked=len(picked),all_converged=bool(all_genes.converged.all()),
                         label_all_bin_correlations=checks.loc[(checks.group=="all")&checks.is_label,"observed_fitted_bin_correlation"].tolist())))


In [ ]:

#10. =========== Load and verify the two formal result files
import io

CG_RESULTS=ROOT  # Set this path independently; use the producer notes above.
CG_VALIDATION=ROOT  # Set this path independently; use the producer notes above.
CG_PANELS=list("CDEFG")
CG_OUTPUT=ROOT  # Set this path independently; use the producer notes above.
CG_RENDER=CG_WORK/"pdf-review"
CG_RUNTIME=ROOT  # Set this path independently; use the producer notes above.
CG_VALIDATOR=ROOT  # Set this path independently; use the producer notes above.
CG_VALIDATOR_PYTHON=CG_RUNTIME/"python/python.exe"
CG_PDFTOPPM=CG_RUNTIME/"native/poppler/Library/bin/pdftoppm.exe"
CG_EXPORT_ROWS=[]
CG_OUTPUT.mkdir(parents=True,exist_ok=True)
require(CG_RESULTS.is_file() and CG_VALIDATION.is_file(),"Both formal C-G result files are required")
CG_CONTRACT=json.loads(CG_VALIDATION.read_text(encoding="utf-8"))
require(digest(CG_RESULTS)==CG_CONTRACT["h5_sha256"],"Formal H5 checksum mismatch")
CG_CACHE={}
with h5py.File(CG_RESULTS,"r") as store:
    require(store.attrs["schema"]=="fig3CG" and store.attrs["schema_version"]==1,"Unsupported formal result schema")
    def check_links(group):
        for key in group:
            require(isinstance(group.get(key,getlink=True),h5py.HardLink),"External/soft HDF5 links are forbidden")
            if isinstance(group[key],h5py.Group):check_links(group[key])
    check_links(store)
    for key,expected in CG_CONTRACT["dataset_manifest"].items():
        ds=store[key]
        require(not ds.is_virtual and ds.external is None,"External or virtual dataset is forbidden")
        value=ds[()]
        require(list(value.shape)==expected["shape"] and value.dtype.str==expected["dtype"],f"Stored axis/dtype mismatch: {key}")
        require(hashlib.sha256(value.tobytes(order="C")).hexdigest()==expected["sha256"],f"Stored bytes changed: {key}")
        CG_CACHE[key]=value

def cg_text(key):
    return CG_CACHE[key].tobytes().decode("utf-8")

def cg_table(key,**kwargs):
    return pd.read_csv(io.StringIO(cg_text(key)),sep="\t" if key.endswith(".tsv") else ",",**kwargs)

CG_LABELS=cg_table("global/Labels.tsv",keep_default_na=False)
CG_SPECIFICITY=cg_table("global/specificity.tsv")
CG_F_SELECTION=cg_table("F_audit/selection.tsv")
CG_F_SELECTION_INFO=json.loads(cg_text("F_audit/selection.json"))
CG_EXPECTED_ROWS={p:CG_CONTRACT["panels"][p]["displayed_genes"] for p in CG_PANELS}
CG_RESULTS_SHA=digest(CG_RESULTS)
CG_VALIDATION_SHA=digest(CG_VALIDATION)
require(STAGES==CG_CONTRACT["render"]["stages"] and STAGE_COLORS==CG_CONTRACT["render"]["stage_colors"],"Stage palette mismatch")
require(PANELS==CG_CONTRACT["render"]["titles"] and PSEUDO_CMAPS==CG_CONTRACT["render"]["pseudotime_cmaps"],"Path palette/title mismatch")
print("Verified formal C-G bundle: float64 fits, exact table bytes, no external HDF5 dependencies.")
